In [ ]:
from google.colab import userdata
KEY_VALUE = 'MY_API_KEY' # userdata.get('MY_KEY')

# MODEL_ID = "Claude-Sonnet-4"
MODEL_ID = "Gemini-2.5-Flash"
API_ENDPOINT="https://api.poe.com/v1"

# Using Google API key via LiteLLM
# MODEL_ID = "gemini/gemini-2.5-flash"

POSTPEND_STRING = '' # '--thinking_budget 0 --web_search false'
GLOBAL_EXECUTOR = 'exec'
MAX_TOKENS = 64000

In [ ]:
!apt-get update
!apt-get install build-essential

In [ ]:
!pip install duckduckgo-search

In [ ]:
!git clone -b v1.23-bp https://github.com/joaopauloschuler/beyond-python-smolagents smolagents

In [ ]:
!pip install ./smolagents[openai]

In [ ]:
import smolagents
from smolagents.bp_tools import *
from smolagents.bp_utils import *
from smolagents.bp_thinkers import *
from smolagents import LogLevel
from smolagents import OpenAIServerModel
from smolagents import VisitWebpageTool, DuckDuckGoSearchTool
from smolagents.default_tools import DuckDuckGoSearchTool

In [ ]:
LocalVistWebPageTool = VisitWebpageTool()
LocalWebSearchTool = DuckDuckGoSearchTool()

In [ ]:
GLOBAL_LOG_LEVEL = LogLevel.DEBUG
STEP_CALLBACKS = [delay_execution_10]

# Via LiteLLM (Gemini, ...)
# model = LiteLLMModel(MODEL_ID, api_key=KEY_VALUE, max_tokens=MAX_TOKENS)

# Using OpenAI protocol
model = OpenAIServerModel(MODEL_ID, api_key=KEY_VALUE, max_tokens=MAX_TOKENS, api_base=API_ENDPOINT)
model.postpend_string = POSTPEND_STRING

model.verbose = False
additional_authorized_imports=['*']
tools = [ run_os_command,
  pascal_interface_to_string, source_code_to_string, string_to_source_code,
  save_string_to_file, load_string_from_file,
  LocalVistWebPageTool, LocalWebSearchTool,
  copy_file, replace_in_file, replace_in_file_from_files, get_file_size,
  get_line_from_file, is_file, print_file_lines
  ]

In [ ]:
!rm advices.notes
!rm *.cpp
!mkdir bin
!rm bin/*
!rm *.txt

In [ ]:
computing_language = "C++"
what_to_code = "task manager"
fileext='.cpp'
start_now = True
refine = True

has_compilation_message = """ When you are asked to compare solutions, compile each version/solution. Only select solutions that do compile.
When compiling code, generate your binaries at the bin/ folder. Do not mix source code with binary (compiled) files.
When testing, review the source code and test if it compiles. Verify for the risk of any infinite loop or memory leak.
Only try to run code after verifying for infinite loop, memory leak and compilation errors.
Feel free to search the internet with error messages if you need.
This is an example how to code and compile a C++ program:
<example>
<savetofile filename='solutionx.cpp'>
#include <iostream>
#include <string>
#include <vector>
// Add other headers as needed

int main() {
    std::cout << "Hello!" << std::endl;
    return 0;
}
</savetofile>
<runcode>
print("Attempting to compile solutionx.cpp...")
compile_output = run_os_command('g++ solutionx.cpp -o bin/task_manager -O2 -std=c++17 -Wall', timeout=120)
print("Compilation output:", compile_output)
# Only attempt to run if compile_output suggests success
if "error:" not in compile_output.lower() and "fatal" not in compile_output.lower():
  if is_file('bin/task_manager'):
    print("Running the compiled program...")
    print(run_os_command('bin/task_manager', timeout=120))
  else:
    print("Executable not found.")
else:
  print("Compilation failed.")
  import re
  error_lines = re.findall(r'solutionx\\.cpp:(\\d+):', compile_output)
  for line_num in set(error_lines): # Use set to avoid duplicate line fetches
    print(f"Error at line {line_num}: {get_line_from_file('solutionx.cpp', int(line_num))}")
</runcode>
</example>

REMEMBER:
* If you have strange compilation errors, you may use get_line_from_file if you like.
* Create a method called self_test. In this method, you will code static inputs for testing (there will be no externally entered data to test with - do not use ReadLn for testing).
* BE BOLD AND CODE AS MANY FEATURES AS YOU CAN!
* If any of your questions is not answered, assume your best guess. Do not keep asking or repeat questions. Just follow your best guess.
* The bin folder has already been created.
* Your goal is C++ coding. Do not spend too much time coding fancy python compilation scripts for C++.
"""

task = """Using only the """+computing_language+""" computing language, code a """+what_to_code+" ."
task += has_compilation_message + """
Feel free to search the internet with error messages if you need.
As you are super-intelligent, I do trust you.
YOU ARE THE BRAIN OF AN AGENT INSIDE OF THE FANTASTIC BEYOND PYTHON SMOLAGENTS: https://github.com/joaopauloschuler/beyond-python-smolagents . Enjoy!
As you are the brain of an agent, this is why you are required to respond with "final_answer" at each conclusive reply from you.
"""

In [ ]:
print(task)

In [ ]:
evolutive_problem_solver(model, task, agent_steps=10, steps=24, start_now=start_now,
    fileext=fileext, tools=tools, log_level=LogLevel.ERROR, step_callbacks=[delay_execution_10], refine=refine)